In [41]:
from ultralytics import YOLO
from roboflow import Roboflow
from dotenv import dotenv_values, load_dotenv
import shutil
import os
import torch


## Get Dataset

In [42]:
secrets = dotenv_values("../.env")

rf = Roboflow(api_key=secrets["ROBOFLOW_API_KEY"])
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolo26")
                

loading Roboflow workspace...
loading Roboflow project...


In [43]:
dataset.location

'/home/roy/dev/projects/ai/football-analysis/training/football-players-detection-1'

## Move Dataset

Have to move test, train etc into football-players-detection-1/football-players-detection-1. It's the format YOLO requires for YOLOV5 newer ones usually don't need it but no harm in doing it apart from disk space.

In [52]:

if not os.path.exists(f"{dataset.location}/football-players-detection-1/train"):
    shutil.copytree(f"{dataset.location}/train", f"{dataset.location}/football-players-detection-1/train")

if not os.path.exists(f"{dataset.location}/football-players-detection-1/test"):
    shutil.copytree(f"{dataset.location}/test", f"{dataset.location}/football-players-detection-1/test")

if not os.path.exists(f"{dataset.location}/football-players-detection-1/valid"):
    shutil.copytree(f"{dataset.location}/valid", f"{dataset.location}/football-players-detection-1/valid")

## Training

In [53]:
import gc

# Optional cleanup to free up memory before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = YOLO("yolo26x.pt")

model.train(data=f"{dataset.location}/data.yaml", epochs=100, imgsz=640, batch=8, workers=4,device=device)

Ultralytics 8.4.33 🚀 Python-3.14.2 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/roy/dev/projects/ai/football-analysis/training/football-players-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train9, nbs=64, nms=False, opset=None,

KeyboardInterrupt: 